In [ ]:
from polyconv_method import (fermion_logZ_numeric, logaddexp,
                             degeneracy_box, level_degeneracy,
                             log_binom, convolve_shell_logspace)

In [ ]:
import numpy as np

# ==========================================
# USER INPUTS: Define your spherical potential
# ==========================================

# Radial Integration Domain
r_min = 1e-6
r_max = 5.0
num_grid = 800

# Maximum angular momentum to compute
l_max = 10

def user_potential_radial(r):
    """
    Define the spherically symmetric potential V(r).
    """
    # --- Default: Regularized Hydrogen Atom ---
    # V(r) = -1 / sqrt(r^2 + a^2)
    a = 0.5
    return -1.0 / np.sqrt(r**2 + a**2)

    # --- Example 2: 3D Isotropic Harmonic Oscillator ---
    # w = 1.0
    # return 0.5 * (w**2) * (r**2)

    # --- Example 3: 3D Spherical Well ---
    # V = np.zeros_like(r)
    # V[r > 2.0] = np.inf
    # return V

# Parameters for testing
n = 5
N = 4
d = None # 3 by definition


In [ ]:
import numpy as np
import scipy.linalg as la
import scipy.special

def get_radial_log_eigenvalues(tau, N, r_min, r_max, num_grid, l_max):
    epsilon = tau / N
    r = np.linspace(r_min, r_max, num_grid)
    dr = (r_max - r_min) / (num_grid - 1)
    
    R, Rp = np.meshgrid(r, r)
    z = R * Rp / epsilon
    
    V_R = user_potential_radial(R)
    V_Rp = user_potential_radial(Rp)
    
    all_log_eigvals = []
    
    # Calculate for each angular momentum l
    for l in range(l_max + 1):
        # Exact 3D radial primitive approximation density matrix
        rho_l = (R * Rp / epsilon) * np.exp(-((R - Rp)**2) / (2 * epsilon)) * scipy.special.ive(l + 0.5, z)
        T_l = np.exp(-epsilon * V_R / 2.0) * rho_l * np.exp(-epsilon * V_Rp / 2.0) * dr
        
        eigvals = la.eigvalsh(T_l)
        eigvals = eigvals[eigvals > 1e-15] # filter numerical noise
        log_eigvals_l = np.log(eigvals)
        
        # Each eigenvalue for angular momentum l has degeneracy 2l + 1
        for log_val in log_eigvals_l:
            all_log_eigvals.append((log_val, 2 * l + 1))
            
    # Sort all states descending by eigenvalue
    all_log_eigvals.sort(key=lambda x: x[0], reverse=True)
    return all_log_eigvals

class NumericalEigenvalueLogW:
    def __init__(self, tau, N, r_min, r_max, num_grid, l_max):
        # list of (log_eigval, degeneracy) tuples
        self.sorted_states = get_radial_log_eigenvalues(tau, N, r_min, r_max, num_grid, l_max)
        self.N = N
        self.max_shell = len(self.sorted_states) - 1
        
    def gk_fn(self, k):
        return self.sorted_states[k][1]
        
    def logwk_fn(self, k, logb):
        return self.sorted_states[k][0] * self.N

def fermion_logZ_numeric_pimc_predict(tau, N, n, d=None, **kwargs):
    num_eig = NumericalEigenvalueLogW(tau, N, r_min, r_max, num_grid, l_max)
    
    kwargs.pop('k_start', None)
    kwargs.pop('max_shell', None)
    kwargs.pop('gk_fn', None)
    kwargs.pop('logwk_fn', None)
    kwargs.pop('Ek_fn', None)
    
    return fermion_logZ_numeric(
        tau=tau, N=N, n=n, d=None, k_start=0, max_shell=num_eig.max_shell,
        gk_fn=num_eig.gk_fn, logwk_fn=num_eig.logwk_fn,
        **kwargs
    )


In [ ]:
from plotting import make_tau_grid, finite_difference_user_sign, plot_logZ_and_fd_multiN

In [ ]:
results = plot_logZ_and_fd_multiN(
    tau_start=1.0,
    tau_end=15.0,
    tau_step=1.0,
    
    n=n,
    d=d,
    N_list=[N],
    show_logZ=False,
    show_fd=True,
    logZ_func=fermion_logZ_numeric_pimc_predict
)


In [ ]:
from benchmarking import (scaling_model, fit_runtime_scaling,
                          plot_runtime_scaling_fit)

In [ ]:
result = plot_runtime_scaling_fit(logZ_func=fermion_logZ_numeric_pimc_predict,
    
    tau=5.0,
    N=2000,
    d=2,
    n_start=0,
    n_end=5000,
    n_step=250,
    repeats=5
)

In [ ]:
result = plot_runtime_scaling_fit(
    tau=0.1,
    N=2000,
    d=2,
    n_start=0,
    n_end=5000,
    n_step=250,
    repeats=3
)

In [ ]:
from benchmarking import (scaling_model_tau_negpow, highest_shell_zero_temp,
                          fit_runtime_vs_tau_negpow, plot_runtime_vs_tau_negpow_fit)

In [ ]:
result = plot_runtime_vs_tau_negpow_fit(logZ_func=fermion_logZ_numeric_pimc_predict,
    
    tau_start=0.1,
    tau_end=20.1,
    tau_step=0.5,
    N=200,
    n=1000,
    d=2,
    repeats=1
)

In [ ]:
from benchmarking import (scaling_model_N_negpow, highest_shell_zero_temp,
                          fit_runtime_vs_N_negpow, plot_runtime_vs_N_negpow_fit)

In [ ]:
result = plot_runtime_vs_N_negpow_fit(logZ_func=fermion_logZ_numeric_pimc_predict,
    
    N_start=1,
    N_end=30,
    N_step=1,
    tau=1.5,
    n=100,
    d=2,
    repeats=100
)